# Unified Results -- Case Study 1 + Case Study 2 (18 experiment x variant runs)

## 1. Fetch repository

In [ ]:
import urllib.request
import zipfile
import sys
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if not REPO_ROOT.exists():
    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = WORKSPACE_ROOT / "repo_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(WORKSPACE_ROOT)

    repo_name = clean_url.split("/")[-1]
    extracted_folder = WORKSPACE_ROOT / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

    zip_path.unlink()

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_ROOT)
print("Source dir:", SRC_DIR)


## 2. Output roots

In [ ]:
DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
OUTPUT_ROOT = DATA_ROOT / "outputs"

EXPERIMENT_ID = "cs1_project_holdout20_innercv_v1"
CS1_EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID
CS2_OUTPUT_ROOT = OUTPUT_ROOT / "case_study_2"

RESULTS_OUTPUT_DIR = OUTPUT_ROOT / "unified_results_summary"
RESULTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CS1_EXPERIMENT_OUTPUT_DIR:", CS1_EXPERIMENT_OUTPUT_DIR)
print("CS2_OUTPUT_ROOT:", CS2_OUTPUT_ROOT)
print("RESULTS_OUTPUT_DIR:", RESULTS_OUTPUT_DIR)


## 3. Experiment matrix definition

In [ ]:
VARIANTS = ["normalized", "abstracted"]

EXPERIMENT_SPECS = [
    {
        "id": "EXP-0",
        "case_study": 1,
        "pipeline": "Logistic Regression",
        "backbone": None,
        "dirs": lambda tag: {
            "dev": OUTPUT_ROOT / f"exp0_idabs_nested_alpha_development_only_v2_reproduction_{tag}",
            "holdout": OUTPUT_ROOT / f"exp0_idabs_nested_alpha_development_only_v2_reproduction_{tag}",
        },
    },
    {
        "id": "EXP-1",
        "case_study": 1,
        "pipeline": "Random Forest",
        "backbone": None,
        "dirs": lambda tag: {
            "dev": CS1_EXPERIMENT_OUTPUT_DIR / f"exp1_rf_idabs_dev_cv_v1_{tag}",
            "holdout": CS1_EXPERIMENT_OUTPUT_DIR / f"exp1_rf_idabs_final_holdout_v1_{tag}",
        },
    },
    {
        "id": "EXP-2",
        "case_study": 1,
        "pipeline": "MLP",
        "backbone": None,
        "dirs": lambda tag: {
            "dev": CS1_EXPERIMENT_OUTPUT_DIR / f"exp2_mlp_idabs_dev_cv_v2_{tag}",
            "holdout": CS1_EXPERIMENT_OUTPUT_DIR / f"exp2_mlp_idabs_final_holdout_v2_{tag}",
        },
    },
    {
        "id": "EXP-3",
        "case_study": 2,
        "pipeline": "Linear Probe",
        "backbone": "CodeBERTa",
        "dirs": lambda tag: {
            "dev": CS2_OUTPUT_ROOT / f"exp3_codeberta_linear_probe_v1_{tag}",
            "holdout": CS2_OUTPUT_ROOT / f"exp3_codeberta_linear_probe_v1_{tag}",
        },
    },
    {
        "id": "EXP-4",
        "case_study": 2,
        "pipeline": "LoRA",
        "backbone": "CodeBERTa",
        "dirs": lambda tag: {
            "dev": CS2_OUTPUT_ROOT / f"exp4_codeberta_lora_v1_{tag}",
            "holdout": CS2_OUTPUT_ROOT / f"exp4_codeberta_lora_v1_{tag}",
        },
    },
    {
        "id": "EXP-5",
        "case_study": 2,
        "pipeline": "HEFT",
        "backbone": "CodeBERTa",
        "dirs": lambda tag: {
            "dev": CS2_OUTPUT_ROOT / f"exp5_codeberta_heft_v1_{tag}",
            "holdout": CS2_OUTPUT_ROOT / f"exp5_codeberta_heft_v1_{tag}",
        },
    },
    {
        "id": "EXP-6",
        "case_study": 2,
        "pipeline": "Linear Probe",
        "backbone": "NeoBERT",
        "dirs": lambda tag: {
            "dev": CS2_OUTPUT_ROOT / f"exp6_neobert_linear_probe_v1_{tag}",
            "holdout": CS2_OUTPUT_ROOT / f"exp6_neobert_linear_probe_v1_{tag}",
        },
    },
    {
        "id": "EXP-7",
        "case_study": 2,
        "pipeline": "LoRA",
        "backbone": "NeoBERT",
        "dirs": lambda tag: {
            "dev": CS2_OUTPUT_ROOT / f"exp7_neobert_lora_v1_{tag}",
            "holdout": CS2_OUTPUT_ROOT / f"exp7_neobert_lora_v1_{tag}",
        },
    },
    {
        "id": "EXP-8",
        "case_study": 2,
        "pipeline": "HEFT",
        "backbone": "NeoBERT",
        "dirs": lambda tag: {
            "dev": CS2_OUTPUT_ROOT / f"exp8_neobert_heft_v1_{tag}",
            "holdout": CS2_OUTPUT_ROOT / f"exp8_neobert_heft_v1_{tag}",
        },
    },
]

print(f"{len(EXPERIMENT_SPECS)} experiments x {len(VARIANTS)} variants = {len(EXPERIMENT_SPECS) * len(VARIANTS)} possible runs.")


## 4. Predictions loader

In [ ]:
import pandas as pd


def load_predictions(directory, pattern):
    matches = sorted(directory.glob(pattern))
    if not matches:
        matches = sorted(directory.glob(pattern.replace(".parquet", ".csv")))
    if not matches:
        raise FileNotFoundError(f"No file matching '{pattern}' found in {directory}")
    if len(matches) > 1:
        raise RuntimeError(f"Multiple files matching '{pattern}' found in {directory}: {matches}")
    path = matches[0]
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


## 5. Discover and load available runs

In [ ]:
records = []
missing = []

for spec in EXPERIMENT_SPECS:
    for tag in VARIANTS:
        dirs = spec["dirs"](tag)
        dev_dir, holdout_dir = dirs["dev"], dirs["holdout"]

        if not dev_dir.exists() or not holdout_dir.exists():
            missing.append((spec["id"], tag, "directory not found"))
            continue

        try:
            dev_predictions = load_predictions(dev_dir, "*oof_predictions*.parquet")
            holdout_predictions = load_predictions(holdout_dir, "*holdout_predictions*.parquet")
        except (FileNotFoundError, RuntimeError) as exc:
            missing.append((spec["id"], tag, str(exc)))
            continue

        records.append({
            "spec": spec,
            "tag": tag,
            "dev_predictions": dev_predictions,
            "holdout_predictions": holdout_predictions,
        })

print(f"Loaded {len(records)} / {len(EXPERIMENT_SPECS) * len(VARIANTS)} experiment x variant combinations.")
print()
print("Loaded:")
for r in records:
    print(f"  - {r['spec']['id']} [{r['tag']}]: dev={len(r['dev_predictions'])} rows, holdout={len(r['holdout_predictions'])} rows")
print()
print("Not yet available:")
for exp_id, tag, reason in missing:
    print(f"  - {exp_id} [{tag}]: {reason}")


## 6. Holdout partition consistency check

In [ ]:
if records:
    reference = records[0]
    reference_projects = set(reference["holdout_predictions"]["project"].unique())
    inconsistent = []
    for r in records[1:]:
        projects = set(r["holdout_predictions"]["project"].unique())
        if projects != reference_projects:
            inconsistent.append(f"{r['spec']['id']} [{r['tag']}]")

    if inconsistent:
        print("WARNING: holdout partition differs from the reference run for:", inconsistent)
        print("Reference:", f"{reference['spec']['id']} [{reference['tag']}]", "with", len(reference_projects), "projects")
    else:
        print(f"All {len(records)} loaded runs share the identical {len(reference_projects)}-project outer holdout.")
else:
    print("No runs loaded yet; nothing to validate.")


## 7. Bootstrap PR-AUC confidence intervals

In [ ]:
import case_study_1.confidence_intervals as confidence_intervals

for r in records:
    r["ci"] = {
        split: confidence_intervals.bootstrap_metric_ci(
            r[f"{split}_predictions"],
            metric="average_precision_pr_auc",
            n_bootstrap=1000,
            random_state=42,
        )
        for split in ("dev", "holdout")
    }
    for split, result in r["ci"].items():
        print(f"{r['spec']['id']} [{r['tag']}] [{split}]")
        print(confidence_intervals.format_ci_report(result))
        print()


## 8. Master results table

In [ ]:
rows = []
for r in records:
    spec = r["spec"]
    row = {
        "experiment": spec["id"],
        "case_study": spec["case_study"],
        "pipeline": spec["pipeline"],
        "backbone": spec["backbone"] or "-",
        "variant": r["tag"],
    }
    for split in ("dev", "holdout"):
        ci = r["ci"][split]
        row[f"{split}_pr_auc"] = ci.point_estimate
        row[f"{split}_ci_low"] = ci.ci_low
        row[f"{split}_ci_high"] = ci.ci_high
    rows.append(row)

results_table = pd.DataFrame(rows).sort_values(["case_study", "experiment", "variant"]).reset_index(drop=True)
results_table.to_csv(RESULTS_OUTPUT_DIR / "unified_results_summary.csv", index=False)
results_table


## 9. Holdout PR-AUC by experiment and variant

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

exp_ids = [s["id"] for s in EXPERIMENT_SPECS]
x = np.arange(len(exp_ids))
width = 0.35
offsets = {"normalized": -width / 2, "abstracted": width / 2}
colors = {"normalized": "tab:blue", "abstracted": "tab:orange"}

fig, ax = plt.subplots(figsize=(14, 6))
for tag in VARIANTS:
    positions = []
    points = []
    err_low = []
    err_high = []
    for i, exp_id in enumerate(exp_ids):
        match = results_table[(results_table["experiment"] == exp_id) & (results_table["variant"] == tag)]
        if match.empty:
            continue
        row = match.iloc[0]
        positions.append(x[i] + offsets[tag])
        points.append(row["holdout_pr_auc"])
        err_low.append(row["holdout_pr_auc"] - row["holdout_ci_low"])
        err_high.append(row["holdout_ci_high"] - row["holdout_pr_auc"])
    ax.bar(positions, points, width, yerr=[err_low, err_high], capsize=4, label=tag, color=colors[tag])

ax.set_xticks(x)
ax.set_xticklabels(exp_ids)
ax.set_ylabel("PR-AUC (outer holdout)")
ax.set_title("Holdout PR-AUC by experiment and preprocessing variant")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_OUTPUT_DIR / "unified_holdout_pr_auc_by_variant.png")
plt.show()


## 10. Precision-recall curves -- Case Study 1

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, tag in zip(axes, VARIANTS):
    for r in records:
        if r["spec"]["case_study"] != 1 or r["tag"] != tag:
            continue
        frame = r["holdout_predictions"]
        precision, recall, _ = precision_recall_curve(frame["label"], frame["y_score"])
        pr_auc = r["ci"]["holdout"].point_estimate
        ax.plot(recall, precision, label=f"{r['spec']['id']} ({r['spec']['pipeline']}, PR-AUC={pr_auc:.4f})")
    ax.set_xlabel("Recall")
    ax.set_title(f"Case Study 1 -- {tag}")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Precision")
plt.tight_layout()
plt.savefig(RESULTS_OUTPUT_DIR / "unified_cs1_pr_curves_holdout.png")
plt.show()


## 11. Precision-recall curves -- Case Study 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, tag in zip(axes, VARIANTS):
    for r in records:
        if r["spec"]["case_study"] != 2 or r["tag"] != tag:
            continue
        frame = r["holdout_predictions"]
        precision, recall, _ = precision_recall_curve(frame["label"], frame["y_score"])
        pr_auc = r["ci"]["holdout"].point_estimate
        label = f"{r['spec']['id']} ({r['spec']['pipeline']}/{r['spec']['backbone']}, PR-AUC={pr_auc:.4f})"
        ax.plot(recall, precision, label=label)
    ax.set_xlabel("Recall")
    ax.set_title(f"Case Study 2 -- {tag}")
    ax.legend(fontsize=7)
axes[0].set_ylabel("Precision")
plt.tight_layout()
plt.savefig(RESULTS_OUTPUT_DIR / "unified_cs2_pr_curves_holdout.png")
plt.show()


## 12. Paired bootstrap comparisons

In [ ]:
import itertools


def find_record(exp_id, tag):
    for r in records:
        if r["spec"]["id"] == exp_id and r["tag"] == tag:
            return r
    return None


def within_group_pairs(exp_ids):
    return [(a, tag, b, tag) for tag in VARIANTS for a, b in itertools.combinations(exp_ids, 2)]


def across_backbone_pairs(method_pairs):
    return [(a, tag, b, tag) for tag in VARIANTS for a, b in method_pairs]


def across_variant_pairs(exp_ids):
    return [(e, "normalized", e, "abstracted") for e in exp_ids]


ALL_PAIRS = (
    within_group_pairs(["EXP-0", "EXP-1", "EXP-2"])
    + within_group_pairs(["EXP-3", "EXP-4", "EXP-5"])
    + within_group_pairs(["EXP-6", "EXP-7", "EXP-8"])
    + across_backbone_pairs([("EXP-3", "EXP-6"), ("EXP-4", "EXP-7"), ("EXP-5", "EXP-8")])
    + across_variant_pairs([s["id"] for s in EXPERIMENT_SPECS])
)

paired_results = []
skipped_pairs = []
for exp_a, tag_a, exp_b, tag_b in ALL_PAIRS:
    record_a = find_record(exp_a, tag_a)
    record_b = find_record(exp_b, tag_b)
    if record_a is None or record_b is None:
        skipped_pairs.append((exp_a, tag_a, exp_b, tag_b))
        continue

    label_a = f"{exp_a} [{tag_a}]"
    label_b = f"{exp_b} [{tag_b}]"
    result = confidence_intervals.paired_bootstrap_metric_ci(
        record_a["holdout_predictions"],
        record_b["holdout_predictions"],
        label_a,
        label_b,
        n_bootstrap=1000,
        random_state=42,
    )
    paired_results.append(result)
    print(confidence_intervals.format_paired_ci_report(result))
    print()

print(f"Computed {len(paired_results)} paired comparisons; skipped {len(skipped_pairs)} (missing run on at least one side).")


## 13. Save paired comparison table

In [ ]:
paired_rows = [
    {
        "comparison_a": result.experiment_a,
        "comparison_b": result.experiment_b,
        "pr_auc_a": result.point_estimate_a,
        "pr_auc_b": result.point_estimate_b,
        "difference": result.point_estimate_diff,
        "ci_low": result.ci_low_diff,
        "ci_high": result.ci_high_diff,
        "significant": (result.ci_low_diff > 0) or (result.ci_high_diff < 0),
    }
    for result in paired_results
]
paired_table = pd.DataFrame(paired_rows)
paired_table.to_csv(RESULTS_OUTPUT_DIR / "unified_paired_holdout_comparisons.csv", index=False)
paired_table


## 14. Does abstraction help? Summary view

In [ ]:
abstraction_rows = paired_table[paired_table["comparison_a"].str.contains(r"\[normalized\]") & paired_table["comparison_b"].str.contains(r"\[abstracted\]")].copy()
abstraction_rows["experiment"] = abstraction_rows["comparison_a"].str.extract(r"^(EXP-\d)")
abstraction_rows = abstraction_rows.sort_values("difference")

if not abstraction_rows.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    y = np.arange(len(abstraction_rows))
    diffs = abstraction_rows["difference"].to_numpy()
    err_low = diffs - abstraction_rows["ci_low"].to_numpy()
    err_high = abstraction_rows["ci_high"].to_numpy() - diffs
    colors = ["tab:green" if s else "tab:gray" for s in abstraction_rows["significant"]]
    ax.errorbar(diffs, y, xerr=[err_low, err_high], fmt="o", capsize=4, ecolor="black", color="black")
    ax.barh(y, diffs, color=colors, alpha=0.4)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(abstraction_rows["experiment"])
    ax.set_xlabel("PR-AUC(abstracted) - PR-AUC(normalized), with 95% CI")
    ax.set_title("Does SCoPE2 abstraction help? (green = CI excludes zero)")
    plt.tight_layout()
    plt.savefig(RESULTS_OUTPUT_DIR / "unified_abstraction_effect.png")
    plt.show()

abstraction_rows


## 15. Save run manifest

In [ ]:
import json
from datetime import datetime, timezone

manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "loaded_runs": [f"{r['spec']['id']} [{r['tag']}]" for r in records],
    "missing_runs": [f"{exp_id} [{tag}]: {reason}" for exp_id, tag, reason in missing],
    "n_loaded": len(records),
    "n_total_possible": len(EXPERIMENT_SPECS) * len(VARIANTS),
    "n_paired_comparisons": len(paired_results),
}

manifest_path = RESULTS_OUTPUT_DIR / "unified_results_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved manifest to", manifest_path)
print(json.dumps(manifest, indent=2))
